# Taller Interactivo: Movimiento de Proyectiles y Aplicaciones en Ingeniería
### Física BR — Física Mecánica
**Profesor:** Juan David Betancur Ríos · **Semestre:** 2026-2

---
**Versión para Google Colab.** Cada punto es autónomo e incluye esquema del montaje,
sliders, botón **"Mostrar solución"** (respuesta oculta hasta que la pidas) y cuestionario.

> **Uso:** `Entorno de ejecución → Ejecutar todo`. Si un slider no aparece, re-ejecuta esa celda.

Gravedad:  g = 9,81 m/s²  (sin resistencia del aire)

---
> © 2026 **Juan David Betancur Ríos**, docente de física para ingeniería. Material educativo de uso académico (Física BR). Todos los derechos reservados.
> Material de uso académico. Todos los derechos reservados. Prohibida su reproducción o distribución sin autorización de los autores.
> *Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico del autor.*

---
## Punto 1 — Lanzamiento de un dron de carga (Ing. Mecánica/Aeronáutica)
Proyectil con rapidez  v₀ = 20,0 m/s  y ángulo  θ = 45°  desde el suelo.

- vₓ = v₀·cos(θ),   v_y = v₀·sen(θ)
- Tiempo de vuelo:  t = 2·v₀·sen(θ)/g
- Alcance:  R = v₀²·sen(2θ)/g       Altura máxima:  H = v₀²·sen²(θ)/(2·g)

> 🔧 **Aplicación en ingeniería:** el lanzamiento de un dron de carga (entregas, agricultura, inspección de ductos) requiere prever su trayectoria para alcanzar el objetivo. El ángulo y la velocidad iniciales determinan el alcance útil.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))


g = 9.81
# --- fin setup ---


def esquema():
    v0=20; th=np.radians(40)
    vx=v0*np.cos(th); vy=v0*np.sin(th)
    fig,ax=plt.subplots(figsize=(6.5,4.2))
    ax.annotate('',xy=(vx,vy),xytext=(0,0),arrowprops=dict(arrowstyle='-|>',color='#c0392b',lw=3))
    ax.text(vx*0.5-1.5,vy*0.5+0.8,'v₀',color='#c0392b',fontsize=14,fontweight='bold')
    ax.annotate('',xy=(vx,0),xytext=(0,0),arrowprops=dict(arrowstyle='-|>',color='#16a085',lw=2.5))
    ax.text(vx*0.5,-1.3,'vₓ = v₀·cos θ',color='#16a085',fontsize=11,ha='center',fontweight='bold')
    ax.annotate('',xy=(vx,vy),xytext=(vx,0),arrowprops=dict(arrowstyle='-|>',color='#e67e22',lw=2.5))
    ax.text(vx+0.4,vy*0.5,'v_y = v₀·sen θ',color='#e67e22',fontsize=11,fontweight='bold')
    ax.plot([vx,vx],[0,vy],color='#e67e22',ls=':',lw=1,alpha=0.6)
    ang=np.linspace(0,th,30); ax.plot(4*np.cos(ang),4*np.sin(ang),color='#34495e',lw=1.2)
    ax.text(4.5,1.3,'θ',fontsize=13,color='#34495e',fontweight='bold')
    ax.annotate('',xy=(vx*0.7,-3),xytext=(vx*0.7,-1),arrowprops=dict(arrowstyle='-|>',color='#2980b9',lw=2.5))
    ax.text(vx*0.7+0.3,-2,'g (constante,\nhacia abajo)',color='#2980b9',fontsize=9)
    ax.axhline(0,color='#8b7355',lw=1.5,alpha=0.5)
    ax.scatter([0],[0],s=120,marker='H',color='#2c3e50',zorder=5,edgecolor='white',linewidth=1.5)
    ax.set_xlim(-2,vx+5); ax.set_ylim(-4,vy+2)
    ax.set_title('Descomposición de la velocidad inicial\n(la horizontal no cambia; la vertical la frena la gravedad)')
    ax.set_xlabel('El movimiento se separa en dos: MRU horizontal + caída vertical',fontsize=9)
    ax.set_yticks([]); ax.set_xticks([])
    ax.spines['left'].set_visible(False); ax.spines['bottom'].set_visible(False)
    plt.tight_layout(); plt.show()
esquema()

def tiro(v0=20.0, ang=45.0):
    th=np.radians(ang); vx=v0*np.cos(th); vy=v0*np.sin(th)
    t_fl=2*vy/g; t=np.linspace(0,t_fl,200); x=vx*t; y=vy*t-0.5*g*t**2
    R=v0**2*np.sin(2*th)/g; H=vy**2/(2*g)
    plt.figure(figsize=(9,4.5)); plt.plot(x,y,lw=2,color='navy')
    plt.scatter([R],[0],color='red',s=60,zorder=5,label=f'alcance={R:.1f} m')
    plt.scatter([R/2],[H],color='green',s=60,zorder=5,label=f'H={H:.1f} m')
    plt.xlabel('x (m)'); plt.ylabel('y (m)'); plt.title(f'v₀={v0:.0f} m/s, θ={ang:.0f}°')
    plt.legend(); plt.grid(alpha=.3); plt.axhline(0,color='k',lw=.5); plt.tight_layout(); plt.show()

interact(tiro,
    v0=FloatSlider(value=20.0,min=5,max=50,step=1,description='v₀ (m/s)'),
    ang=FloatSlider(value=45.0,min=10,max=80,step=5,description='θ (grados)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)

    v0=20.0; th=np.radians(45)
    _pc(f"vₓ={v0*np.cos(th):.2f} m/s, v_y={v0*np.sin(th):.2f} m/s")
    _pc(f"Tiempo de vuelo = {2*v0*np.sin(th)/g:.3f} s")
    _pc(f"Alcance = {v0**2*np.sin(2*th)/g:.3f} m")
    _pc(f"Altura máxima = {(v0*np.sin(th))**2/(2*g):.3f} m")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("Un proyectil se lanza a 20,0 m/s con un ángulo de 45,0°. Encuentre su tiempo de vuelo total.",2.883,0.04,"s","t=2v₀senθ/g≈2,88 s.", id_preg="Proyecti_p1_1", peso=2.0)
quiz_numerico("Para ese proyectil lanzado a 20,0 m/s y 45,0°, encuentre su alcance horizontal.",40.775,0.04,"m","R=v₀²sen(2θ)/g≈40,8 m.", id_preg="Proyecti_p1_2", peso=2.0)
quiz_opcion_multiple("En la trayectoria de un proyectil, ¿en qué punto la componente vertical de la velocidad es cero?",
    ["Al inicio","En la altura máxima","Al caer","Nunca"],1,"En el punto más alto v_y=0.", id_preg="Proyecti_p1_3", peso=2.0)

# --- Estilo visual Física BR (apariencia global de los esquemas) ---
try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


---
## Punto 2 — Entrega desde un dron en vuelo (Ing. Civil/Logística)
Paquete soltado con velocidad **horizontal**  v₀ = 15,0 m/s  desde  h = 45,0 m.

- Tiempo de caída:  t = √(2·h/g)
- Alcance:  x = v₀·t       Velocidad de impacto:  v = √(v₀² + (g·t)²)

> 🔧 **Aplicación en ingeniería:** soltar carga desde un dron en vuelo (paquetes, insumos agrícolas) exige calcular dónde caerá, considerando la velocidad del dron. Es logística aérea aplicada, cada vez más usada en zonas de difícil acceso.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))


g = 9.81
# --- fin setup ---


def esquema():
    fig,ax=plt.subplots(figsize=(6.8,4.2))
    # eje horizontal: MRU
    ax.annotate('',xy=(6,4),xytext=(0,4),arrowprops=dict(arrowstyle='-|>',color='#16a085',lw=2.5))
    ax.text(3,4.3,'HORIZONTAL: velocidad constante (MRU)',color='#16a085',fontsize=10,ha='center',fontweight='bold')
    for i in range(1,4):
        ax.annotate('',xy=(1.5*i+1.5,4),xytext=(1.5*i,4),arrowprops=dict(arrowstyle='->',color='#16a085',lw=1,alpha=0.5))
    # eje vertical: caída acelerada
    ax.annotate('',xy=(0,0),xytext=(0,3.5),arrowprops=dict(arrowstyle='-|>',color='#e67e22',lw=2.5))
    ax.text(-0.3,1.7,'VERTICAL: caída\nacelerada (g)',color='#e67e22',fontsize=10,rotation=90,va='center',fontweight='bold')
    for i,y in enumerate([3.0,2.3,1.3,0.0]):
        sz=0.3+i*0.2
        ax.annotate('',xy=(0.15,y),xytext=(0.15,y+sz),arrowprops=dict(arrowstyle='->',color='#e67e22',lw=1,alpha=0.6))
    # mensaje central
    ax.text(3.3,2,'Los dos movimientos\nson INDEPENDIENTES\ny ocurren a la vez',ha='center',fontsize=10,
            color='#2c3e50',bbox=dict(boxstyle='round',fc='#f8f9fa',ec='#bdc3c7'))
    ax.scatter([0],[4],s=180,marker='H',color='#2c3e50',zorder=5,edgecolor='white',linewidth=1.5)
    ax.text(0,4.5,'dron',fontsize=8,ha='center',color='#2c3e50')
    ax.set_xlim(-1.5,7); ax.set_ylim(-0.5,5)
    ax.set_title('Lanzamiento horizontal: dos movimientos simultáneos e independientes')
    ax.set_xticks([]); ax.set_yticks([])
    ax.spines['left'].set_visible(False); ax.spines['bottom'].set_visible(False)
    plt.tight_layout(); plt.show()
esquema()

def caida(v0=15.0, h=45.0):
    t_fall=np.sqrt(2*h/g); t=np.linspace(0,t_fall,200); x=v0*t; y=h-0.5*g*t**2
    plt.figure(figsize=(9,4.5)); plt.plot(x,y,lw=2,color='darkgreen')
    plt.scatter([v0*t_fall],[0],color='red',s=60,zorder=5,label=f'impacto x={v0*t_fall:.1f} m')
    plt.xlabel('x (m)'); plt.ylabel('y (m)'); plt.title(f'v₀={v0:.0f} m/s desde h={h:.0f} m')
    plt.legend(); plt.grid(alpha=.3); plt.axhline(0,color='k',lw=.5); plt.tight_layout(); plt.show()

interact(caida,
    v0=FloatSlider(value=15.0,min=5,max=40,step=1,description='v₀ (m/s)'),
    h=FloatSlider(value=45.0,min=10,max=100,step=5,description='h (m)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)

    v0=15.0; h=45.0; t=np.sqrt(2*h/g)
    _pc(f"Tiempo de caída = √(2h/g) = {t:.3f} s")
    _pc(f"Alcance horizontal = v₀·t = {v0*t:.3f} m")
    _pc(f"Velocidad de impacto = {np.hypot(v0,g*t):.3f} m/s")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 2 ---")
quiz_numerico("Un objeto se lanza horizontalmente desde una altura de 45,0 m. Encuentre el tiempo que tarda en caer al suelo.",3.029,0.04,"s","t=√(2h/g)≈3,03 s. No depende de v₀.", id_preg="Proyecti_p2_4", peso=2.0)
quiz_numerico("Para ese lanzamiento horizontal a 15,0 m/s desde 45,0 m de altura, encuentre el alcance horizontal.",45.434,0.04,"m","x=v₀t≈45,4 m.", id_preg="Proyecti_p2_5", peso=2.0)
quiz_opcion_multiple("En un lanzamiento horizontal, si se aumenta la velocidad horizontal inicial, ¿qué ocurre con el tiempo de caída?",
    ["Aumenta","Disminuye","No cambia: la caída vertical es independiente de v₀","Se duplica"],2,
    "El tiempo de caída solo depende de h.", id_preg="Proyecti_p2_6", peso=2.0)


---
## Punto 3 — Ángulo óptimo de alcance (Ing. Industrial/Deportiva)
Con rapidez fija  v₀ = 20,0 m/s, se busca el ángulo de mayor alcance.

**Alcance:**  R = v₀²·sen(2θ)/g  → máximo cuando 2θ=90°, es decir θ=45°.

> 🔧 **Aplicación en ingeniería:** maximizar el alcance (45° sin resistencia) aparece en aspersión de cultivos, sistemas de riego y balística de proyectos. Conocer el ángulo óptimo ahorra energía y mejora la cobertura.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))


g = 9.81
# --- fin setup ---


def esquema():
    g=9.81; v0=20
    fig,ax=plt.subplots(figsize=(7,4.2))
    ax.axhspan(0,12,color='#eaf4fb',alpha=0.6,zorder=0)
    ax.axhspan(-0.5,0,color='#d9c8a3',alpha=0.6,zorder=0)
    ax.axhline(0,color='#8b7355',lw=1.5,zorder=1)
    colores=['#3498db','#9b59b6','#e74c3c','#27ae60','#f39c12']
    angulos=[15,30,45,60,75]
    for a,col in zip(angulos,colores):
        th=np.radians(a); t_v=2*v0*np.sin(th)/g; t=np.linspace(0,t_v,80)
        x=v0*np.cos(th)*t; y=v0*np.sin(th)*t-0.5*g*t**2
        lw=3.5 if a==45 else 1.8
        alpha=1.0 if a==45 else 0.65
        ax.plot(x,y,color=col,lw=lw,alpha=alpha,label=f'{a}°'+(' (máx)' if a==45 else ''),zorder=3 if a==45 else 2)
    ax.set_xlabel('distancia horizontal (m)'); ax.set_ylabel('altura (m)')
    ax.set_title('Ángulo óptimo de alcance · 45° maximiza la distancia')
    ax.legend(title='ángulo',framealpha=0.9,fontsize=8,loc='upper right')
    ax.set_ylim(-0.5,12); ax.grid(alpha=0.2,linestyle='--')
    plt.tight_layout(); plt.show()
esquema()

def alcance_vs_angulo(v0=20.0, ang_marcado=45.0):
    angs=np.linspace(0,90,300); R=v0**2*np.sin(2*np.radians(angs))/g
    Rm=v0**2*np.sin(2*np.radians(ang_marcado))/g
    plt.figure(figsize=(9,4.5)); plt.plot(angs,R,lw=2,color='purple')
    plt.axvline(45,color='green',ls='--',label='óptimo 45°')
    plt.scatter([ang_marcado],[Rm],color='red',s=60,zorder=5,label=f'θ={ang_marcado:.0f}°→{Rm:.1f} m')
    plt.xlabel('ángulo θ (grados)'); plt.ylabel('alcance R (m)'); plt.title(f'Alcance vs ángulo (v₀={v0:.0f} m/s)')
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

interact(alcance_vs_angulo,
    v0=FloatSlider(value=20.0,min=5,max=50,step=1,description='v₀ (m/s)'),
    ang_marcado=FloatSlider(value=45.0,min=5,max=85,step=5,description='θ (grados)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)

    v0=20.0
    _pc(f"Alcance máximo (θ=45°) = v₀²/g = {v0**2/g:.3f} m")
    _pc("Ángulos complementarios (ej. 30° y 60°) dan el mismo alcance.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("Encuentre el alcance de un proyectil lanzado a 20,0 m/s con el ángulo que maximiza la distancia (45,0°).",40.775,0.04,"m","R_max=v₀²/g≈40,8 m.", id_preg="Proyecti_p3_7", peso=2.0)
quiz_opcion_multiple("Despreciando la resistencia del aire, ¿qué ángulo de lanzamiento produce el alcance horizontal máximo?",["30°","45°","60°","90°"],1,
    "sen(2θ)=1 ⟹ θ=45°.", id_preg="Proyecti_p3_8", peso=2.0)
quiz_opcion_multiple("¿Qué par de ángulos de lanzamiento producen el mismo alcance horizontal?",
    ["Ninguno","Complementarios (ej. 30° y 60°)","Solo 45°","Iguales"],1,
    "θ y 90°−θ dan igual alcance.", id_preg="Proyecti_p3_9", peso=2.0)


---
## Punto 4 — Velocidad y posición en un instante (Ing. Mecánica/Control)
Proyectil con  v₀ = 25,0 m/s,  θ = 50°.  Se analiza el estado (posición y velocidad)
en un instante intermedio  t = 1,50 s  (útil para control y seguimiento de trayectorias).

- Posición:  x = vₓ·t,   y = v_y·t − ½·g·t²
- Velocidad:  vₓ constante,   v_y(t) = v₀·sen(θ) − g·t,   |v| = √(vₓ² + v_y²)

> 🔧 **Aplicación en ingeniería:** conocer la velocidad y posición en un instante permite programar la interceptación, el seguimiento o la entrega precisa en sistemas automatizados (drones, brazos robóticos, aspersores). Es control de movimiento en tiempo real.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))


g = 9.81
# --- fin setup ---


def esquema():
    fig,ax=plt.subplots(figsize=(7,4.2))
    # mostrar el vector velocidad en 3 momentos: subida, cima, bajada
    momentos=[(1.5,3.0,'subida'),(4.0,3.6,'cima'),(6.5,3.0,'bajada')]
    vx=2.0
    vys=[1.8,0.0,-1.8]
    for (px,py,lbl),vy in zip(momentos,vys):
        # vx siempre igual (verde)
        ax.annotate('',xy=(px+vx,py),xytext=(px,py),arrowprops=dict(arrowstyle='-|>',color='#16a085',lw=2.2))
        # vy cambia (naranja)
        if abs(vy)>0.1:
            ax.annotate('',xy=(px,py+vy),xytext=(px,py),arrowprops=dict(arrowstyle='-|>',color='#e67e22',lw=2.2))
        # resultante (rojo)
        ax.annotate('',xy=(px+vx,py+vy),xytext=(px,py),arrowprops=dict(arrowstyle='-|>',color='#c0392b',lw=2))
        ax.scatter([px],[py],s=70,color='#2c3e50',zorder=5,edgecolor='white',linewidth=1.5)
        ax.text(px,py-0.6,lbl,ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    # trayectoria de fondo (tenue, solo referencia)
    xt=np.linspace(0,8,50); yt=3.0+1.8*np.sin(np.pi*xt/8)-0.0
    ax.plot(xt,yt+0.3,color='#bdc3c7',lw=1.5,ls='--',alpha=0.6,zorder=0)
    # leyenda de colores
    ax.text(0.5,0.7,'vₓ constante',color='#16a085',fontsize=9,fontweight='bold')
    ax.text(0.5,0.3,'v_y cambia con g',color='#e67e22',fontsize=9,fontweight='bold')
    ax.text(3.5,0.5,'v resultante',color='#c0392b',fontsize=9,fontweight='bold')
    ax.set_xlim(-0.5,9); ax.set_ylim(0,5.5)
    ax.set_title('Cómo cambia la velocidad durante el vuelo\n(vₓ nunca cambia; v_y disminuye, se anula y crece hacia abajo)')
    ax.set_xticks([]); ax.set_yticks([])
    ax.spines['left'].set_visible(False); ax.spines['bottom'].set_visible(False)
    plt.tight_layout(); plt.show()
esquema()

def instante(v0=25.0, ang=50.0, t_inst=1.5):
    th=np.radians(ang); vx=v0*np.cos(th); vy0=v0*np.sin(th)
    t_fl=2*vy0/g; t=np.linspace(0,t_fl,200); x=vx*t; y=vy0*t-0.5*g*t**2
    xi=vx*t_inst; yi=vy0*t_inst-0.5*g*t_inst**2; vyi=vy0-g*t_inst; vi=np.hypot(vx,vyi)
    plt.figure(figsize=(9,4.5)); plt.plot(x,y,lw=2,color='navy',alpha=.6)
    plt.scatter([xi],[yi],color='red',s=70,zorder=5,label=f't={t_inst:.1f}s: ({xi:.1f}, {yi:.1f}) m')
    plt.quiver(xi,yi,vx,vyi,color='red',scale=80,width=.006,label=f'|v|={vi:.1f} m/s')
    plt.xlabel('x (m)'); plt.ylabel('y (m)'); plt.title(f'v₀={v0:.0f} m/s, θ={ang:.0f}°, instante t={t_inst:.1f}s')
    plt.legend(); plt.grid(alpha=.3); plt.axhline(0,color='k',lw=.5); plt.tight_layout(); plt.show()

interact(instante,
    v0=FloatSlider(value=25.0,min=10,max=50,step=1,description='v₀ (m/s)'),
    ang=FloatSlider(value=50.0,min=10,max=80,step=5,description='θ (grados)'),
    t_inst=FloatSlider(value=1.5,min=0.2,max=4,step=0.1,description='t (s)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)

    v0=25.0; th=np.radians(50); ti=1.5
    vx=v0*np.cos(th); vy0=v0*np.sin(th); vyi=vy0-g*ti
    _pc(f"Posición: x={vx*ti:.2f} m, y={vy0*ti-0.5*g*ti**2:.2f} m")
    _pc(f"Velocidad: vₓ={vx:.2f} m/s (constante), v_y={vyi:.2f} m/s")
    _pc(f"Rapidez |v| = √(vₓ²+v_y²) = {np.hypot(vx,vyi):.2f} m/s")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 4 ---")
quiz_numerico("Un proyectil se lanza a 25,0 m/s con 50,0° de elevación. Encuentre su altura en el instante t = 1,50 s.",17.69,0.05,"m",
    "y=v₀senθ·t−½gt²≈17,7 m.", id_preg="Proyecti_p4_10", peso=2.0)
quiz_numerico("Para ese proyectil (25,0 m/s, 50,0°), encuentre su rapidez en el instante t = 1,50 s.",16.67,0.05,"m/s",
    "vₓ=16,07, v_y=4,44 ⟹ |v|≈16,7 m/s.", id_preg="Proyecti_p4_11", peso=2.0)
quiz_opcion_multiple("En el movimiento de un proyectil (sin resistencia del aire), ¿qué componente de la velocidad permanece constante?",
    ["La vertical v_y","La horizontal vₓ (no hay fuerza horizontal)","Ninguna","Ambas"],1,
    "Sin aire, no hay aceleración horizontal, así que vₓ es constante.", id_preg="Proyecti_p4_12", peso=2.0)


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))

# Calificación formativa del taller
calificacion_final(total_preguntas=12)


---
## ✅ Fin del taller de Proyectiles (4 puntos)
Experimenta con los sliders y **predice** antes de pulsar "Mostrar solución".
*Física Mecánica — 2026-2.*